# EEG_09b — Ablation Study: GAT Domain Adversarial × Metodi Grafo

Questo notebook analizza i risultati dell'esperimento EEG_09, confrontando **5 metodi di costruzione del grafo** con le 3 architetture GAT (con e senza domain adversarial training).

## Modelli testati

| Modello | Architettura | Adversarial | Note |
|---------|-------------|-------------|------|
| `GAT_2L` | 2× GATConv | No | Baseline GAT |
| `GAT_2L_ADV` | 2× GATConv + GRL | Si | DAGAM-style, λ=0.5 |
| `GAT_3L_ADV` | 3× GATConv + GRL | Si | Versione più profonda |

## Metodi grafo

| Metodo | Descrizione |
|--------|-------------|
| PCC | Pearson Correlation Coefficient k-NN (k=6) |
| PLV | Phase Locking Value (theta+alpha 4–13 Hz) |
| wPLI | Weighted Phase Lag Index (robusto a volume conduction) |
| Learned | Matrice adiacenza apprendibile end-to-end |
| Dynamic | PCC per-finestra (8 finestre temporali) |

## Dati
I risultati vengono caricati dai CSV generati da EEG_09 (`data/interim/eeg09_gat_*_results.csv`).
Se i CSV non esistono ancora (EEG_09 ancora in esecuzione), il notebook mostra un avviso e salta i plot.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

# ---------------------------------------------------------------------------
# Configurazione
# ---------------------------------------------------------------------------
N_CLASSES  = 4          # schema di clustering (deve corrispondere a EEG_09)
CHANCE     = 1.0 / N_CLASSES

# Trova la root del progetto tramite .git
_here = Path(".").resolve()
project_root = _here
for p in [_here, *_here.parents]:
    if (p / ".git").exists():
        project_root = p
        break
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Figures dir:  {figures_dir}")
print(f"Chance level: {CHANCE:.1%}")

# ---------------------------------------------------------------------------
# Carica risultati da tutti i CSV dei metodi
# ---------------------------------------------------------------------------
dfs = []
for method in ["pcc", "plv", "wpli", "learned", "dynamic"]:
    csv_path = project_root / "data" / "interim" / f"eeg09_gat_{N_CLASSES}_{method}_results.csv"
    if csv_path.exists():
        df_m = pd.read_csv(csv_path)
        # Aggiungi colonna method se non presente (retrocompatibilità)
        if "graph_method" not in df_m.columns:
            df_m["graph_method"] = method
        dfs.append(df_m)
        print(f"  [{method.upper():7s}] {len(df_m)} modelli caricati da {csv_path.name}")
    else:
        print(f"  [{method.upper():7s}] CSV non trovato: {csv_path.name}")

if dfs:
    df = pd.concat(dfs, ignore_index=True)
    # Normalizza nome metodo per display
    method_display = {"pcc": "PCC", "plv": "PLV", "wpli": "wPLI",
                      "learned": "Learned", "dynamic": "Dynamic"}
    df["method"] = df["graph_method"].map(method_display).fillna(df["graph_method"])
    df["time_min"] = df.get("time_s", pd.Series(dtype=float)) / 60.0
    print(f"\nTotale record: {len(df)}")
    print(df[["model", "method", "val_bacc", "test_bacc"]].to_string(index=False))
else:
    print("\nNessun risultato disponibile. Esegui EEG_09 prima (loop automatico metodi).")
    df = pd.DataFrame()

## Plot 1 — Ablation: val_bacc per Metodo × Architettura

Il grafico mostra la **balanced accuracy** (val_bacc) per ogni combinazione metodo-architettura GAT.
La linea tratteggiata rossa indica il **chance level** (25% per 4 classi).
I modelli con GRL adversariale (`_ADV`) dovrebbero superare il baseline `GAT_2L`.

In [ ]:
if df.empty:
    print("Dati non ancora disponibili. Esegui EEG_09 prima.")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

    METHOD_COLORS = {
        "PCC":     "#4C72B0",
        "PLV":     "#DD8452",
        "wPLI":    "#55A868",
        "Learned": "#C44E52",
        "Dynamic": "#8172B2",
    }
    MODEL_ORDER = ["GAT_2L", "GAT_2L_ADV", "GAT_3L_ADV"]
    methods_present = [m for m in ["PCC", "PLV", "wPLI", "Learned", "Dynamic"]
                       if m in df["method"].values]

    x = np.arange(len(MODEL_ORDER))
    width = 0.15
    offsets = np.linspace(-(len(methods_present)-1)*width/2,
                           (len(methods_present)-1)*width/2,
                           len(methods_present))

    fig, ax = plt.subplots(figsize=(12, 5), dpi=150)

    for offset, method in zip(offsets, methods_present):
        values = []
        for mname in MODEL_ORDER:
            row = df[(df["model"] == mname) & (df["method"] == method)]
            values.append(row["val_bacc"].values[0] if len(row) > 0 else np.nan)
        bars = ax.bar(x + offset, values, width, label=method,
                      color=METHOD_COLORS.get(method, "gray"), alpha=0.85)
        # Annotazioni valore
        for bar, v in zip(bars, values):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=6.5, rotation=90)

    ax.axhline(CHANCE, color="red", linestyle="--", linewidth=1.5,
               label=f"Chance ({CHANCE:.1%})")
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_ORDER, fontsize=11)
    ax.set_ylabel("val_bacc", fontsize=11)
    ax.set_title(
        f"EEG_09 — val_bacc per Metodo Grafo × Architettura GAT ({N_CLASSES} classi)",
        fontsize=12, pad=12
    )
    ax.legend(title="Metodo Grafo", fontsize=9, title_fontsize=9)
    ax.set_ylim(0, max(df["val_bacc"].max() * 1.15, CHANCE * 1.3))
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    out_path = figures_dir / "eeg09_ablation_graph_methods.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato in: {out_path}")

## Plot 2 — val_acc vs val_bacc: Class Collapse

Scatter plot che mette in evidenza il fenomeno di **class collapse**: alcuni modelli raggiungono una
`val_acc` elevata (predizione della classe maggioritaria) ma una `val_bacc` a chance level.
I modelli adversariali (`_ADV`) dovrebbero avere sia `val_acc` che `val_bacc` più equilibrate.

In [ ]:
if df.empty:
    print("Dati non ancora disponibili. Esegui EEG_09 prima.")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

    METHOD_COLORS = {
        "PCC":     "#4C72B0",
        "PLV":     "#DD8452",
        "wPLI":    "#55A868",
        "Learned": "#C44E52",
        "Dynamic": "#8172B2",
    }
    MODEL_MARKERS = {
        "GAT_2L":     "o",
        "GAT_2L_ADV": "^",
        "GAT_3L_ADV": "s",
    }

    fig, ax = plt.subplots(figsize=(9, 6), dpi=150)

    for _, row in df.iterrows():
        color  = METHOD_COLORS.get(row["method"], "gray")
        marker = MODEL_MARKERS.get(row["model"], "o")
        ax.scatter(row["val_acc"], row["val_bacc"], color=color, s=100, zorder=5,
                   edgecolors="white", linewidths=0.6, marker=marker)
        short_name = row["model"].replace("GAT_", "")
        ax.annotate(
            short_name,
            xy=(row["val_acc"], row["val_bacc"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=7.5,
            color="black",
            alpha=0.85,
        )

    # Linee di chance level
    ax.axvline(CHANCE, color="gray", linestyle="--", linewidth=1.2, alpha=0.7,
               label=f"Chance (val_acc = {CHANCE:.2f})")
    ax.axhline(CHANCE, color="tomato", linestyle="--", linewidth=1.2, alpha=0.7,
               label=f"Chance (val_bacc = {CHANCE:.2f})")

    # Legenda metodi
    legend_patches = [
        mpatches.Patch(color=c, label=m) for m, c in METHOD_COLORS.items()
        if m in df["method"].values
    ]
    # Legenda marker architetture
    from matplotlib.lines import Line2D
    marker_legend = [
        Line2D([0], [0], marker=mk, color="w", markerfacecolor="gray",
               markersize=9, label=mn)
        for mn, mk in MODEL_MARKERS.items()
    ]
    legend1 = ax.legend(handles=legend_patches, title="Metodo Grafo", loc="upper left",
                        fontsize=9, title_fontsize=9)
    ax.add_artist(legend1)
    legend2 = ax.legend(handles=marker_legend, title="Architettura", loc="lower right",
                        fontsize=9, title_fontsize=9)
    ax.add_artist(legend2)
    ax.legend(loc="upper right", fontsize=8)

    ax.set_xlabel("val_acc (accuratezza semplice)", fontsize=11)
    ax.set_ylabel("val_bacc (balanced accuracy)", fontsize=11)
    ax.set_title(
        "val_acc vs val_bacc — EEG_09 GAT Adversarial\n"
        "(modelli con alta val_acc ma bacc=chance collassano sulla classe maggioritaria)",
        fontsize=11, pad=12
    )

    plt.tight_layout()
    out_path = figures_dir / "eeg09_acc_vs_bacc.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato in: {out_path}")

## Plot 3 — Heatmap: val_bacc per Metodo × Architettura

La heatmap mostra come la balanced accuracy vari al variare del metodo di costruzione del grafo
(righe) e dell'architettura GAT (colonne). Celle verdi indicano performance sopra chance (25%).

In [ ]:
if df.empty:
    print("Dati non ancora disponibili. Esegui EEG_09 prima.")
else:
    METHODS = ["PCC", "PLV", "wPLI", "Learned", "Dynamic"]
    ARCHS   = ["GAT_2L", "GAT_2L_ADV", "GAT_3L_ADV"]

    # Costruisci la matrice (NaN dove la combinazione non esiste ancora)
    heatmap_data = pd.DataFrame(np.nan, index=METHODS, columns=ARCHS)
    for _, row in df.iterrows():
        if row["method"] in METHODS and row["model"] in ARCHS:
            heatmap_data.loc[row["method"], row["model"]] = row["val_bacc"]

    vmin = max(heatmap_data.min().min() * 0.98, CHANCE * 0.9) if not heatmap_data.isna().all().all() else 0.20
    vmax = max(heatmap_data.max().max() * 1.02, CHANCE * 1.2) if not heatmap_data.isna().all().all() else 0.40

    fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
    sns.heatmap(
        heatmap_data,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        center=CHANCE,
        vmin=vmin,
        vmax=vmax,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "val_bacc", "shrink": 0.8},
        annot_kws={"size": 10},
    )
    ax.set_title(
        f"Heatmap val_bacc — Metodo Grafo × Architettura GAT ({N_CLASSES} classi)\n"
        f"(verde = sopra chance {CHANCE:.2f}, rosso = sotto)",
        fontsize=11, pad=12
    )
    ax.set_xlabel("Architettura GAT", fontsize=11)
    ax.set_ylabel("Metodo Costruzione Grafo", fontsize=11)
    ax.tick_params(axis="x", labelsize=10)
    ax.tick_params(axis="y", labelsize=10, rotation=0)

    plt.tight_layout()
    out_path = figures_dir / "eeg09_heatmap_bacc.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato in: {out_path}")

## Plot 4 — Tempo di Training per Metodo × Architettura

I modelli adversariali (`_ADV`) richiedono più epoch per convergere (`PATIENCE_ADV=40` vs `PATIENCE=15`).
I metodi **Learned** e **Dynamic** sono più lenti per la costruzione on-the-fly del grafo.

In [ ]:
if df.empty:
    print("Dati non ancora disponibili. Esegui EEG_09 prima.")
elif "time_s" not in df.columns or df["time_s"].isna().all():
    print("Colonna time_s non disponibile nel CSV — saltato.")
else:
    METHOD_COLORS = {
        "PCC":     "#4C72B0",
        "PLV":     "#DD8452",
        "wPLI":    "#55A868",
        "Learned": "#C44E52",
        "Dynamic": "#8172B2",
    }

    df_time = df.copy()
    df_time["time_min"] = df_time["time_s"] / 60.0
    df_time["label"] = df_time["method"] + " / " + df_time["model"]
    df_time = df_time.sort_values("time_min", ascending=True)

    labels = df_time["label"].tolist()
    times  = df_time["time_min"].values
    colors = [METHOD_COLORS.get(m, "gray") for m in df_time["method"]]

    fig, ax = plt.subplots(figsize=(12, max(5, len(labels) * 0.35)), dpi=150)
    bars = ax.barh(labels, times, color=colors, edgecolor="white", height=0.65)

    # Annotazioni valore
    for bar, t in zip(bars, times):
        ax.text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f"{t:.1f} min",
            va="center", ha="left", fontsize=8.5,
        )

    # Linea di riferimento: media metodi veloci
    fast_df = df_time[df_time["method"].isin(["PCC", "PLV", "wPLI"])]
    if len(fast_df) > 0:
        avg_fast = fast_df["time_min"].mean()
        ax.axvline(avg_fast, color="steelblue", linestyle=":", linewidth=1.5,
                   label=f"Media PCC/PLV/wPLI ({avg_fast:.1f} min)")

    # Legenda metodi
    legend_patches = [
        mpatches.Patch(color=c, label=m) for m, c in METHOD_COLORS.items()
        if m in df_time["method"].values
    ]
    ax.legend(handles=legend_patches, fontsize=8, loc="lower right")

    ax.set_xlabel("Tempo di training (minuti)", fontsize=11)
    ax.set_title(
        f"Tempo di Training — EEG_09 GAT Adversarial ({N_CLASSES} classi)\n"
        "(Learned e Dynamic più lenti per costruzione grafo on-the-fly)",
        fontsize=11, pad=12
    )
    ax.set_xlim(0, times.max() * 1.15 if len(times) > 0 else 10)
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout()
    out_path = figures_dir / "eeg09_training_time.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato in: {out_path}")

## Plot 5 — Struttura Grafi: Stesso Trial × 5 Metodi

Visualizzazione comparativa dei **5 grafi di connettività** costruiti dallo stesso trial EEG
(soggetto 0, epoca 0) con i 5 metodi.
Nota: richiede i dataset pre-calcolati da EEG_07b (`data/interim/graphs/`).

In [ ]:
import math

def read_eloc(path):
    """Legge posizioni elettrodi dal file .locs (formato EEGLAB)."""
    pos = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                try:
                    theta  = float(parts[1])
                    radius = float(parts[2])
                    name   = parts[3]
                    th = math.radians(theta)
                    pos[name] = (radius * math.sin(th), radius * math.cos(th))
                except Exception:
                    pass
    return pos


def get_region_color(name):
    """Colore per regione cerebrale basato sul prefisso del nome elettrodo."""
    n = name.upper()
    if n.startswith("F"):
        return "#4C72B0"   # frontale — blu
    elif n.startswith("T"):
        return "#55A868"   # temporale — verde
    elif n.startswith("P"):
        return "#DD8452"   # parietale — arancio
    elif n.startswith("O"):
        return "#C44E52"   # occipitale — rosso
    elif n.startswith("C"):
        return "#8172B2"   # centrale — viola
    else:
        return "#999999"   # altro — grigio


try:
    import torch
    from torch_geometric.data import Data

    locs_path  = project_root / "src" / "io" / "ebneuro.locs"
    graphs_dir = project_root / "data" / "interim" / "graphs"

    if not locs_path.exists():
        raise FileNotFoundError(f"File .locs non trovato: {locs_path}")
    if not graphs_dir.exists():
        raise FileNotFoundError(f"Cartella grafi non trovata: {graphs_dir}")

    # Leggi posizioni elettrodi
    eloc = read_eloc(locs_path)

    # Dataset da caricare (nome file → etichetta display)
    datasets = {
        "PCC":     "dataset_pcc_k6.pt",
        "PLV":     "dataset_plv_k6.pt",
        "wPLI":    "dataset_wpli_k6.pt",
        "Learned": "dataset_learned_k6.pt",
        "Dynamic": "dataset_dynamic_k6.pt",
    }

    fig, axes = plt.subplots(1, 5, figsize=(20, 4.5), dpi=150)
    fig.suptitle(
        "Struttura del Grafo per lo Stesso Trial (soggetto 0, epoca 0) × 5 Metodi",
        fontsize=12, y=1.02
    )

    for ax, (method, fname) in zip(axes, datasets.items()):
        fpath = graphs_dir / fname
        ax.set_title(method, fontsize=11, fontweight="bold")
        ax.set_aspect("equal")
        ax.axis("off")

        if not fpath.exists():
            ax.text(0.5, 0.5, f"File non trovato:\n{fname}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="gray")
            continue

        data_list = torch.load(fpath, map_location="cpu", weights_only=False)
        sample = data_list[0] if isinstance(data_list, list) else data_list

        # Nomi elettrodi nell'ordine del tensore (esclusi A1/A2)
        all_names   = list(eloc.keys())
        valid_names = [n for n in all_names if n not in ("A1", "A2")]
        n_nodes     = sample.edge_index.max().item() + 1
        node_names  = valid_names[:n_nodes]

        xs = np.array([eloc.get(n, (0, 0))[0] for n in node_names])
        ys = np.array([eloc.get(n, (0, 0))[1] for n in node_names])
        node_colors = [get_region_color(n) for n in node_names]

        # Disegna archi
        ei = sample.edge_index.numpy()
        edge_weight = None
        if hasattr(sample, "edge_attr") and sample.edge_attr is not None:
            ew = sample.edge_attr.numpy().flatten()
            if len(ew) == ei.shape[1]:
                ew_min, ew_max = ew.min(), ew.max()
                if ew_max > ew_min:
                    edge_weight = 0.3 + 1.7 * (ew - ew_min) / (ew_max - ew_min)
                else:
                    edge_weight = np.ones(len(ew)) * 0.8

        for i in range(ei.shape[1]):
            src, dst = ei[0, i], ei[1, i]
            lw = float(edge_weight[i]) if edge_weight is not None else 0.6
            ax.plot(
                [xs[src], xs[dst]], [ys[src], ys[dst]],
                color="gray", alpha=0.35, linewidth=lw, zorder=1
            )

        # Disegna nodi
        ax.scatter(xs, ys, c=node_colors, s=28, zorder=5,
                   edgecolors="white", linewidths=0.4)

    # Legenda regioni
    region_legend = [
        mpatches.Patch(color="#4C72B0", label="Frontale (F)"),
        mpatches.Patch(color="#55A868", label="Temporale (T)"),
        mpatches.Patch(color="#DD8452", label="Parietale (P)"),
        mpatches.Patch(color="#C44E52", label="Occipitale (O)"),
        mpatches.Patch(color="#8172B2", label="Centrale (C)"),
        mpatches.Patch(color="#999999", label="Altro"),
    ]
    fig.legend(handles=region_legend, loc="lower center", ncol=6, fontsize=8,
               bbox_to_anchor=(0.5, -0.06))

    plt.tight_layout()
    out_path = figures_dir / "eeg09_graph_comparison_trial0.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato in: {out_path}")

except FileNotFoundError as e:
    print(f"[SKIP] Dataset non disponibili localmente: {e}")
    print("Questa cella produce output solo dove i dataset pre-calcolati sono disponibili.")
    print("Esegui EEG_07b_precompute_graphs.ipynb prima di rieseguire questa cella.")
except Exception as e:
    print(f"[ERRORE] {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

## Conclusioni

### Riepilogo Risultati EEG_09 (da completare dopo il training)

| Metodo | Architettura | val_bacc | test_bacc | Adversarial | Tempo (min) |
|--------|-------------|----------|-----------|-------------|-------------|
| PCC | GAT_2L | — | — | No | — |
| PCC | GAT_2L_ADV | — | — | Si | — |
| PCC | GAT_3L_ADV | — | — | Si | — |
| PLV | GAT_2L | — | — | No | — |
| PLV | GAT_2L_ADV | — | — | Si | — |
| PLV | GAT_3L_ADV | — | — | Si | — |
| wPLI | GAT_2L | — | — | No | — |
| wPLI | GAT_2L_ADV | — | — | Si | — |
| wPLI | GAT_3L_ADV | — | — | Si | — |
| Learned | GAT_2L | — | — | No | — |
| Learned | GAT_2L_ADV | — | — | Si | — |
| Learned | GAT_3L_ADV | — | — | Si | — |
| Dynamic | GAT_2L | — | — | No | — |
| Dynamic | GAT_2L_ADV | — | — | Si | — |
| Dynamic | GAT_3L_ADV | — | — | Si | — |

### Domande chiave

1. **Il domain adversarial training aiuta?** I modelli `_ADV` superano `GAT_2L` in val_bacc?
2. **Il metodo del grafo fa differenza?** Come in EEG_08, ci aspettiamo che non cambi molto.
3. **Il class collapse è ridotto?** I `_ADV` dovrebbero avere val_acc e val_bacc più vicini.
4. **Trade-off tempo/accuratezza**: i modelli adversariali sono più lenti ma migliorano la generalizzazione?

### Prossimi Passi

- **EEG_10 (HGNN)**: Hypergraph Neural Networks — obiettivo principale della tesi
- **Contrastive learning** subject-invariant (Shen et al. 2022)
- **Instance Normalization** cross-soggetto (Bomatter et al. 2024 — 1 riga di codice)